In [17]:
import pandas as pd
df1 = pd.read_csv("dataset/train/train_source1.tsv", sep="\t")

In [18]:
df2 = pd.read_csv("dataset/train/train_source2.tsv", sep="\t")
df3 = pd.read_csv("dataset/train/train_source3.tsv", sep="\t")
test_df1 = pd.read_csv("dataset/test/test_source1.tsv", sep="\t")
test_df2 = pd.read_csv("dataset/test/test_source2.tsv", sep="\t")
test_df3 = pd.read_csv("dataset/test/test_source3.tsv", sep="\t")

In [19]:
df1.isnull().sum()

entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

In [20]:
df2.isnull().sum()

entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64

In [21]:
df3.isnull().sum()

entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64

In [22]:
import pandas as pd
import unicodedata
import re
from anyascii import anyascii


In [23]:
# ============================================================
# 1. REGEX PATTERNS & NORMALIZATION LOOKUPS
# ============================================================

LEGAL_SUFFIXES_REGEX = re.compile(
    r"\b("
    r"pvt\s+ltd|"
    r"private\s+limited|"
    r"limited|"
    r"ltd|"
    r"llc|"
    r"llp|"
    r"incorporated|"
    r"inc|"
    r"corporation|"
    r"corp|"
    r"sarl|"
    r"sasu|"
    r"sas|"
    r"eurl|"
    r"sci|"
    r"snc"
    r")\b",
    re.IGNORECASE
)

URL_PREFIX_REGEX = re.compile(r"https?://|www\.", re.IGNORECASE)
DOMAIN_REGEX = re.compile(r"\.(com|org|net|in|fr|co|io|biz|info|gov|us)\b", re.IGNORECASE)

STOPWORDS = {"of", "the", "and", "at", "for", "in", "a", "an", "by", "de", "la", "le", "les", "du", "des"}

COUNTRY_MAP = {
    "us": "us", "usa": "us", "united states": "us",
    "india": "in", "ind": "in",
    "france": "fr", "fr": "fr"
}

ADDRESS_ABBREVIATIONS = {
    # US / UK / India
    r"\brd\.?\b": "road",
    r"\bst\.?\b": "street",
    r"\bave?\.?\b": "avenue",
    r"\bdr\.?\b": "drive",
    r"\bblvd\.?\b": "boulevard",
    r"\bln\.?\b": "lane",
    r"\bct\.?\b": "court",
    r"\bhwy\.?\b": "highway",
    r"\bpkwy\.?\b": "parkway",
    # Unit / apartment
    r"\bapt\.?\b": "unit",
    r"\bapartment\b": "unit",
    r"\bste\.?\b": "unit",
    r"\bsuite\b": "unit",
    r"\bpmb\b": "unit",
    # France
    r"\br\.(?=\s|$)": "rue",
    r"\bbd\.?\b": "boulevard",
    r"\bbvd\b": "boulevard",
    r"\ball\.?\b": "allee",
    r"\bimp\.?\b": "impasse",
    r"\bpl\.?\b": "place",
    r"\bche?\.?\b": "chemin",
    r"\brte\b": "route"
}

ORDINALS = {
    r"\b1st\b": "1", r"\bfirst\b": "1",
    r"\b2nd\b": "2", r"\bsecond\b": "2",
    r"\b3rd\b": "3", r"\bthird\b": "3",
    r"\b4th\b": "4", r"\bfourth\b": "4",
    r"\b5th\b": "5", r"\bfifth\b": "5"
}

US_STATES = {
    "al": "alabama", "ak": "alaska", "az": "arizona", "ar": "arkansas",
    "ca": "california", "co": "colorado", "ct": "connecticut", "de": "delaware",
    "fl": "florida", "ga": "georgia", "hi": "hawaii", "id": "idaho",
    "il": "illinois", "in": "indiana", "ia": "iowa", "ks": "kansas",
    "ky": "kentucky", "la": "louisiana", "me": "maine", "md": "maryland",
    "ma": "massachusetts", "mi": "michigan", "mn": "minnesota", "ms": "mississippi",
    "mo": "missouri", "mt": "montana", "ne": "nebraska", "nv": "nevada",
    "nh": "new hampshire", "nj": "new jersey", "nm": "new mexico", "ny": "new york",
    "nc": "north carolina", "nd": "north dakota", "oh": "ohio", "ok": "oklahoma",
    "or": "oregon", "pa": "pennsylvania", "ri": "rhode island", "sc": "south carolina",
    "sd": "south dakota", "tn": "tennessee", "tx": "texas", "ut": "utah",
    "vt": "vermont", "va": "virginia", "wa": "washington", "wv": "west virginia",
    "wi": "wisconsin", "wy": "wyoming", "dc": "district of columbia"
}

US_STATE_REGEX = re.compile(r"\b(" + "|".join(sorted(US_STATES, key=len, reverse=True)) + r")\b")


In [24]:
# ============================================================
# 2. FIELD-LEVEL CLEANING FUNCTIONS
# ============================================================

def clean_business_name(text: str) -> str:
    """
    Transliterates non-Latin scripts to ASCII, lowercases, removes
    URLs/domains, legal suffixes, punctuation, and stopwords.
    """
    if not isinstance(text, str) or not text.strip() or text.strip().lower() == "nan":
        return ""

    # 1. Unicode normalization and transliteration to ASCII
    s = unicodedata.normalize("NFKC", text)
    s = anyascii(s).lower()

    # 2. Remove URL prefixes and domain extensions
    s = URL_PREFIX_REGEX.sub("", s)
    s = DOMAIN_REGEX.sub(" ", s)

    # 3. Remove legal / corporate suffixes
    s = LEGAL_SUFFIXES_REGEX.sub(" ", s)

    # 4. Remove punctuation
    s = re.sub(r"[^\w\s]", " ", s)

    # 5. Remove stopwords and normalize whitespace
    tokens = [w for w in s.split() if w not in STOPWORDS]
    return " ".join(tokens)


def clean_address_text(addr: str, country_code: str = "") -> str:
    """
    Transliterates address to ASCII, normalizes street abbreviations,
    expands US state abbreviations (if country is US), standardizes ordinals,
    and removes punctuation while PRESERVING postal/PIN code digits.
    """
    if not isinstance(addr, str) or not addr.strip() or addr.strip().lower() == "nan":
        return ""

    # 1. Unicode normalization and transliteration to ASCII
    s = unicodedata.normalize("NFKC", addr)
    s = anyascii(s).lower()

    # 2. Standardize street abbreviations
    for pattern, repl in ADDRESS_ABBREVIATIONS.items():
        s = re.sub(pattern, repl, s)

    # 3. Standardize ordinals (1st -> 1, 2nd -> 2, etc.)
    for pattern, repl in ORDINALS.items():
        s = re.sub(pattern, repl, s)

    # 4. Expand US states ONLY for US addresses (avoids Indian address collision where 'in' -> 'indiana')
    if country_code in ("us", "usa"):
        s = US_STATE_REGEX.sub(lambda m: US_STATES[m.group(1)], s)

    # 5. Remove punctuation but KEEP all alphanumeric characters (postal codes / PIN codes / digits preserved)
    s = re.sub(r"[^\w\s]", " ", s)

    # 6. Normalize unit noise
    s = re.sub(r"\bunit\s+\w+\b", " ", s)

    # 7. Normalize whitespace
    return re.sub(r"\s+", " ", s).strip()


In [25]:
# Sanity check on sample names and addresses
test_names = [
    "Raj Investments LLP",
    "ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி",
    "राज इन्वेस्टमेंट्स प्राइवेट लिमिटेड",
    "Café & Crème Résidence",
    "Dréxkor",
    "wilfordhancock.com"
]
print("--- Business Name Normalization Test ---")
for name in test_names:
    print(f"{name:<35} -> {clean_business_name(name)}")

test_addresses = [
    ("1064 Newton Rd, Unit 11, Iowa City, IA 52242", "us"),
    ("797, Lake Town Block A, Kolkata, Howrah 700089, West Bengal", "in"),
    ("63 R. DE DIEPPE, LILLE 59000, Hauts-de-France", "fr"),
    ("H.No.16-11-23/37/A, Flat No.207, Near Fortis Hospital, Hyderabad", "in")
]
print("\n--- Address Normalization Test (Postal codes preserved in clean_address) ---")
for addr, country in test_addresses:
    print(f"{addr[:45]:<45} -> {clean_address_text(addr, country)}")


--- Business Name Normalization Test ---
Raj Investments LLP                 -> raj investments
ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி     -> raj investments elelpi
राज इन्वेस्टमेंट्स प्राइवेट लिमिटेड -> raj investmemts praivet
Café & Crème Résidence              -> cafe creme residence
Dréxkor                             -> drexkor
wilfordhancock.com                  -> wilfordhancock

--- Address Normalization Test (Postal codes preserved in clean_address) ---
1064 Newton Rd, Unit 11, Iowa City, IA 52242  -> 1064 newton road iowa city iowa 52242
797, Lake Town Block A, Kolkata, Howrah 70008 -> 797 lake town block a kolkata howrah 700089 west bengal
63 R. DE DIEPPE, LILLE 59000, Hauts-de-France -> 63 rue de dieppe lille 59000 hauts de france
H.No.16-11-23/37/A, Flat No.207, Near Fortis  -> h no 16 11 23 37 a flat no 207 near fortis hospital hyderabad


In [26]:
# ============================================================
# 3. DATASET PROCESSOR
# ============================================================

def process_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalizes country, business_name, and business_address in-place.
    - Uses clean_address (NOT address_clean)
    - Preserves postal code numbers inside clean_address
    - Does NOT create postal_code or combined_key columns
    """
    print(f"Normalizing dataset with {len(df):,} rows...")

    # 1. Clean country code
    country_values = df["country"].fillna("").astype(str).str.lower().str.strip()
    df["country_clean"] = country_values.map(COUNTRY_MAP).fillna(country_values)

    # 2. Clean business name
    df["clean_name"] = df["business_name"].fillna("").apply(clean_business_name)

    # 3. Clean business address (preserves postal code within clean_address)
    df["clean_address"] = [
        clean_address_text(addr, c)
        for addr, c in zip(df["business_address"], df["country_clean"])
    ]

    # 4. Remove any legacy or unwanted columns
    df.drop(columns=["postal_code", "combined_key", "address_clean"], inplace=True, errors="ignore")

    return df


In [27]:
# ============================================================
# 4. APPLY NORMALIZATION TO TRAIN AND TEST DATASETS
# ============================================================

for data in (df1, df2, df3, test_df1, test_df2, test_df3):
    process_dataset(data)

print("Train and test normalization complete!")


Normalizing dataset with 2,206,821 rows...
Normalizing dataset with 5,034,616 rows...
Normalizing dataset with 5,285,603 rows...
Normalizing dataset with 1,732,544 rows...
Normalizing dataset with 4,887,273 rows...
Normalizing dataset with 5,082,316 rows...
Train and test normalization complete!


In [28]:
# Verify final schema and cleaned records on df1
df1.head(10)


,entity_id,business_name,business_address,country,country_clean,clean_name,clean_address
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,us,orelee s barbershop,1795 westchester drive high point north carolina
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,us,prime money,17560 ellis road tahlequah oklahoma
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,us,b retail,1712 montebello avenue phoenix arizona
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,us,christ chapel,2100 cameron drive g dundalk maryland
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,in,prabhav business center,797 lake town block a kolkata howrah west bengal
5,S1-851869949,Custom Wealth Services LLC,"OH, Columbus, 5559 Orville Avenue",US,us,custom wealth services,ohio columbus 5559 orville avenue
6,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India,in,consulting nyasa nursing,2505 tower 1 oakwood runwal greens mulund gore...
7,S1-27541239,Nexus Anchor Rain,"1111 Church Street, Unit 2007, Nashville, TN",US,us,nexus anchor rain,1111 church street nashville tennessee
8,S1-629417405,Moore Bitwise Inc,"337 Oakland Avenue, Michigan City, IN",US,us,moore bitwise,337 oakland avenue michigan city indiana
9,S1-22305073,Dermatology Green Medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",US,us,dermatology green medicine,294 meadowcreek drive 2 village of pewaukee wi...


In [29]:
test_df1.head(10)


,entity_id,business_name,business_address,country,country_clean,clean_name,clean_address
0,S1-714132312,Zephay Labs Inc,"2621 Cotten Road, Tyler, TX",US,us,zephay labs,2621 cotten road tyler texas
1,S1-106407869,Vision Partners Corp,"IA, Iowa City, 1064 Newton Rd, Unit 11",US,us,vision partners,iowa iowa city 1064 newton road
2,S1-156285671,<< Team Ecole,"175 Boulevard du Président Franklin Roosevelt,...",France,fr,team ecole,175 boulevard du president franklin roosevelt ...
3,S1-689823050,Red Perfect Trading,"Mirzapur, Ews 12, Uttar Pradesh, Mirzapursadar...",India,in,red perfect trading,mirzapur ews 12 uttar pradesh mirzapursadar aw...
4,S1-921369899,ZNB Club SARL,"Nouvelle-Aquitaine, La Teste-de-Buch, 5 bis Ru...",France,fr,znb club,nouvelle aquitaine la teste de buch 5 bis rue ...
5,S1-481669221,Nandlal Kisan LLP,"D-61 Ifs Apartmentmayur Vihar I, New Delhi, Ea...",India,in,nandlal kisan,d 61 ifs apartmentmayur vihar i new delhi east...
6,S1-909865979,Cure Seafood,"1325 Brooklyn Walk, Issaquah, WA",US,us,cure seafood,1325 brooklyn walk issaquah washington
7,S1-280204013,Om Constructions Pvt Ltd,"Karauli, Rajasthan, Karauli, Pani Ki Tanki Ke ...",India,in,om constructions,karauli rajasthan karauli pani ki tanki ke pas...
8,S1-742053041,Roongta Sangh,"Bhubaneswar, Sub Plot No.-L6/29, Mahodadhi Bha...",India,in,roongta sangh,bhubaneswar sub plot no l6 29 mahodadhi bhawan...
9,S1-913506265,Thermal & Fils SASU,"20 Rue Parmentier, Dunkerque, Hauts-de-France",France,fr,thermal fils,20 rue parmentier dunkerque hauts de france
